# Pipeline — fase 2: hidratação seletiva e rotulagem

Orquestra a **fase 2** do trabalho sobre os resultados finais da fase 1
(`notebooks/pipeline.ipynb`): para **todos os eventos de uma vez**, seleciona os tweets a
hidratar por cluster (D6), mantém o banco de cache (`data/database/hydrated.sqlite`) e, em
seguida, hidratará e rotulará. Cada etapa tem uma célula de inspeção.

Etapas: **Banco** (schema + eventos) → **Migração do legado** (hidratação de maio/2026) →
**M8** seleção por cluster → **Consolidação** (plano de hidratação + gravação no banco) →
**M9** hidratação *(a definir)* → **Status**.

**Idempotência.** Toda célula pode ser re-executada sem efeito colateral acumulado:
- estágios (`.run(...)`) usam o cache por existência de arquivo da fase 1 (`FORCE_FROM`);
- o banco só recebe *upserts* (tweets, autores, eventos) ou *replace por evento* (seleção);
- a migração grava apenas o que ainda não está no cache, com o `hydrated_at` real do snapshot.

**Pré-requisitos por evento:** `graph_nodes.parquet` + `graph_edges.parquet` + `run_config.json`
em `data/processed/<evento>/`. O `retweets.parquet` (M1) é regenerado dos CSVs brutos se faltar.
**Nenhuma chamada à API do X é feita por este notebook ainda** — M9 está em aberto.

In [1]:
import sys
from pathlib import Path

# Raiz do projeto: o notebook vive em notebooks/, mas pode rodar de notebooks/
# (Jupyter) ou da raiz do repositório (nbconvert/CI).
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))  # habilita `from modules.* import ...`

import json
import pandas as pd

from modules.database import Database
from modules.load import RetweetLoader
from modules.select_tweets import TopTweetSelector, load_final_graph
from modules.fetch_x_data import COST_PER_TWEET, COST_PER_USER

## Configuração

A fase 2 roda sobre **todos** os eventos, porque a hidratação deduplica IDs entre eles e o
custo é global. Os parâmetros da seleção são os do D6 revisado (2026-09-10).

In [2]:
# ---- Eventos (subpastas de data/processed/) ----
EVENTOS = ["mobilizacao-0709", "roberto-jefferson", "eleicoes", "invasao-3-poderes"]

# ---- Parâmetros da seleção (D6, revisão 2026-09-10) ----
K = 100                   # top-K por cluster
K_SMALL = 20              # K para clusters com pouco peso interno
MIN_FRAC = 0.01           # cluster entra se tiver ≥ 1% dos nós (= min_frac da fase 1)
SMALL_WEIGHT_FRAC = 0.05  # ≤ 5% do peso TOTAL do grafo em arestas internas → K_SMALL

# ---- Cache por estágio ----
# FORCE_FROM = None -> usa cache onde houver.  FORCE_FROM = 8 -> recomputa M8 (e posteriores).
# Mudou K, MIN_FRAC ou o grafo de algum evento? Aponte FORCE_FROM para o módulo afetado.
FORCE_FROM = None

def _force(n: int) -> bool:
    return FORCE_FROM is not None and n >= FORCE_FROM

DB_PATH = PROJECT_ROOT / "data" / "database" / "hydrated.sqlite"
RAW = {ev: PROJECT_ROOT / "data" / "raw" / ev for ev in EVENTOS}
PROCESSED = {ev: PROJECT_ROOT / "data" / "processed" / ev for ev in EVENTOS}

for ev in EVENTOS:
    for f in ("graph_nodes.parquet", "graph_edges.parquet", "run_config.json"):
        assert (PROCESSED[ev] / f).exists(), f"{ev}: falta {f} — rode notebooks/pipeline.ipynb para esse evento"
    cfg = json.loads((PROCESSED[ev] / "run_config.json").read_text())
    print(f"{ev:20s} N={cfg['min_user_retweets']:<2} τ={cfg['tau']} min_frac={cfg['min_frac']} "
          f"gerado em {cfg['generated_at']}")
print(f"\nBanco:   {DB_PATH}")
print(f"Seleção: K={K}, K_SMALL={K_SMALL}, MIN_FRAC={MIN_FRAC}, SMALL_WEIGHT_FRAC={SMALL_WEIGHT_FRAC}")

mobilizacao-0709     N=3  τ=0.1 min_frac=0.01 gerado em 2026-06-28T04:17:37
roberto-jefferson    N=8  τ=0.1 min_frac=0.01 gerado em 2026-06-28T17:49:17
eleicoes             N=5  τ=0.1 min_frac=0.01 gerado em 2026-06-28T18:03:26
invasao-3-poderes    N=7  τ=0.1 min_frac=0.01 gerado em 2026-06-28T04:07:16

Banco:   /home/vinicius/tcc/data/database/hydrated.sqlite
Seleção: K=100, K_SMALL=20, MIN_FRAC=0.01, SMALL_WEIGHT_FRAC=0.05


## Banco — conexão, schema e eventos

`Database` garante o schema de forma idempotente (`CREATE TABLE IF NOT EXISTS`; ver
`data/database/README.md`). A tabela `events` é re-semeada por *upsert* com o `slug` igual ao
nome da pasta em `data/processed/` — o slug antigo `democracia-3010` (eleicoes) é removido.

In [3]:
db = Database(DB_PATH)

EVENTS_ROWS = [
    {"slug": "mobilizacao-0709",  "name": "Mobilização do 7 de setembro de 2022",
     "event_date": "2022-09-07", "notes": "raw: data/raw/mobilizacao-0709/ (0709_mobilizacao.csv)"},
    {"slug": "roberto-jefferson", "name": "Caso Roberto Jefferson",
     "event_date": "2022-10-23", "notes": "raw: data/raw/roberto-jefferson/ (2310_robertojefferson_*.csv)"},
    {"slug": "eleicoes",          "name": "Debate sobre democracia no dia do 2º turno",
     "event_date": "2022-10-30", "notes": "raw: data/raw/eleicoes/ (3010_democracia.csv); slug antigo no banco: democracia-3010"},
    {"slug": "invasao-3-poderes", "name": "Ataques de 8 de janeiro de 2023",
     "event_date": "2023-01-08", "notes": "raw: data/raw/invasao-3-poderes/ (0801/0901_invasao-*.csv)"},
]
db.upsert_events(EVENTS_ROWS)
with db.conn:
    db.conn.execute("DELETE FROM events WHERE slug = 'democracia-3010'")   # no-op se já não existir

print("Eventos:")
for r in db.conn.execute("SELECT slug, event_date, name FROM events ORDER BY event_date"):
    print(f"  {r['slug']:20s} {r['event_date']}  {r['name']}")
print("\nLinhas por tabela:", db.table_counts())

Eventos:
  mobilizacao-0709     2022-09-07  Mobilização do 7 de setembro de 2022
  roberto-jefferson    2022-10-23  Caso Roberto Jefferson
  eleicoes             2022-10-30  Debate sobre democracia no dia do 2º turno
  invasao-3-poderes    2023-01-08  Ataques de 8 de janeiro de 2023

Linhas por tabela: {'author_classification': 0, 'community_membership': 0, 'event_top_tweets': 1340, 'events': 4, 'tweets': 83, 'users': 72}


## Migração do legado — hidratação de maio/2026

Antes do banco existir, o `fetch_x_data.py` gravou em JSON a primeira hidratação (invasão dos
3 Poderes, top-100 **global** — critério anterior ao D6 revisado): 100 IDs pedidos, **83 tweets
e 72 autores** retornados (17% de atrição, ver D6). Esse material é cache pago: entra em
`tweets`/`users` com o `hydrated_at` **do snapshot original**, não o de hoje. A célula só grava
o que ainda não está no banco, então re-executar não muda nada.

In [4]:
LEGACY_DIR = PROCESSED["invasao-3-poderes"]
LEGACY_HYDRATED_AT = "2026-05-09T00:57:00-03:00"   # mtime original dos JSON (git não preserva mtime)

def _jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

legacy_tweets = _jsonl(LEGACY_DIR / "hydrated_tweets.jsonl")
with open(LEGACY_DIR / "hydrated_users.json", encoding="utf-8") as f:
    legacy_users = list(json.load(f).values())

before = db.table_counts()
cached_t, cached_u = db.cached_tweet_ids(), db.cached_user_ids()
new_t = [t for t in legacy_tweets if str(t["id"]) not in cached_t]
new_u = [u for u in legacy_users if str(u["id"]) not in cached_u]
db.upsert_tweets(new_t, hydrated_at=LEGACY_HYDRATED_AT)
db.upsert_users(new_u, hydrated_at=LEGACY_HYDRATED_AT)
after = db.table_counts()

print(f"Legado nos JSON:  {len(legacy_tweets)} tweets, {len(legacy_users)} autores")
print(f"Migrados agora:   {len(new_t)} tweets, {len(new_u)} autores")
print(f"tweets: {before['tweets']} -> {after['tweets']}   users: {before['users']} -> {after['users']}")

Legado nos JSON:  83 tweets, 72 autores
Migrados agora:   0 tweets, 0 autores
tweets: 83 -> 83   users: 72 -> 72


In [5]:
# Inspeção da migração — autores dos tweets em cache que não estão em `users` são dado
# faltante REAL (conta suspensa/removida na hidratação de maio), não erro (README, princípio 4).
sem_autor = db.conn.execute("""
    SELECT COUNT(*) FROM tweets t LEFT JOIN users u ON u.user_id = t.author_id
    WHERE u.user_id IS NULL""").fetchone()[0]
print(f"tweets em cache cujo autor não está em users: {sem_autor}")
print(pd.read_sql("SELECT tweet_id, author_id, lang, retweet_count, substr(text,1,70) AS texto, hydrated_at "
                  "FROM tweets ORDER BY retweet_count DESC LIMIT 5", db.conn).to_string(index=False))

tweets em cache cujo autor não está em users: 0
           tweet_id           author_id lang  retweet_count                                                                    texto               hydrated_at
1612169657154256901  904095261689073664   pt          28348 bolsonaristas invadindo o congresso \n\na pm de brasilia https://t.co/c2 2026-05-09T00:57:00-03:00
1612164676145610757          3120745861   pt          27780   o bolsonaro na disney em orlando e os corno quebrando tudo em brasilia 2026-05-09T00:57:00-03:00
1612191803901444098          2670726740   pt          22958   Vocês devem ter acompanhado a barbárie em Brasília hoje. Aquelas pesso 2026-05-09T00:57:00-03:00
1612251119266271233 1003810164749856769   pt          21549   Obras de arte que foram DESTRUÍDAS ou ROUBADAS por bolsonaristas em Br 2026-05-09T00:57:00-03:00
1612193089095073793            72189315   pt          17414   Presidente da Galoucura se manifestou e se colocou disponível para ir  2026-05-09T00:57:00-03:0

## Módulo 8 — Seleção por cluster (D6)

Para cada evento: comunidades com ≥ `MIN_FRAC` dos nós; ranking pelo nº de **membros do cluster**
que retuitaram (usuário×tweet distinto conta 1); `K` por cluster, ou `K_SMALL` quando o peso das
arestas internas do cluster é ≤ `SMALL_WEIGHT_FRAC` do peso **total** do grafo. Desempate
determinístico (`rt_graph` desc, `tweet_id` asc). Persiste `top_tweets.parquet` +
`top_tweets_stats.json` em `data/processed/<evento>/`.

O mesmo tweet pode aparecer em mais de um cluster — a deduplicação é feita na consolidação.

In [6]:
SELECOES, STATS = {}, {}
for ev in EVENTOS:
    print(f"\n== {ev}")
    nodes, edges = load_final_graph(PROCESSED[ev])
    retweets = RetweetLoader(RAW[ev]).run(out_dir=PROCESSED[ev])        # M1: regenera se faltar
    sel = TopTweetSelector(k=K, k_small=K_SMALL, min_frac=MIN_FRAC,
                           small_weight_frac=SMALL_WEIGHT_FRAC)
    SELECOES[ev] = sel.run(nodes, edges, retweets, out_dir=PROCESSED[ev], force=_force(8))
    STATS[ev] = s = sel.stats
    print(f"   {s['n_nodes']:,} nós | {len(s['clusters'])} clusters selecionados "
          f"(+{s['n_excluded_clusters']} abaixo de {MIN_FRAC:.0%}, {s['excluded_nodes_frac']:.1%} dos nós) | "
          f"{s['n_slots']} slots -> {s['n_unique_ids']} IDs únicos ({s['n_overlap']} repetidos entre clusters)")


== mobilizacao-0709


[cache] RetweetLoader: hit


[cache] TopTweetSelector: hit
   16,495 nós | 4 clusters selecionados (+6 abaixo de 1%, 0.8% dos nós) | 320 slots -> 224 IDs únicos (96 repetidos entre clusters)

== roberto-jefferson


[cache] RetweetLoader: hit


[cache] TopTweetSelector: hit
   27,042 nós | 3 clusters selecionados (+1 abaixo de 1%, 0.1% dos nós) | 300 slots -> 217 IDs únicos (83 repetidos entre clusters)

== eleicoes


[cache] RetweetLoader: hit
[cache] TopTweetSelector: hit
   9,742 nós | 5 clusters selecionados (+1 abaixo de 1%, 0.0% dos nós) | 420 slots -> 181 IDs únicos (239 repetidos entre clusters)

== invasao-3-poderes


[cache] RetweetLoader: hit


[cache] TopTweetSelector: hit
   33,305 nós | 3 clusters selecionados (+8 abaixo de 1%, 0.1% dos nós) | 300 slots -> 251 IDs únicos (49 repetidos entre clusters)


In [7]:
# Inspeção M8 — um cluster por linha, todos os eventos
rows = []
for ev in EVENTOS:
    for c, cs in STATS[ev]["clusters"].items():
        rows.append({"evento": ev, "cluster": int(c), "nós": cs["n_nodes"],
                     "% nós": 100 * cs["frac_nodes"], "% peso interno": 100 * cs["intra_weight_frac"],
                     "K": cs["k"], "candidatos": cs["n_candidates"], "selecionados": cs["n_selected"]})
tabela_clusters = pd.DataFrame(rows)
print(tabela_clusters.to_string(index=False, formatters={"% nós": "{:.1f}".format,
                                                          "% peso interno": "{:.2f}".format,
                                                          "nós": "{:,}".format,
                                                          "candidatos": "{:,}".format}))

print("\nTop-3 de cada cluster (invasao-3-poderes):")
print(SELECOES["invasao-3-poderes"].groupby("community").head(3).to_string(index=False))

           evento  cluster    nós % nós % peso interno   K candidatos  selecionados
 mobilizacao-0709        0  7,525  45.6          26.49 100      3,393           100
 mobilizacao-0709        1  5,251  31.8          19.96 100      2,436           100
 mobilizacao-0709        2  3,154  19.1          33.01 100      1,854           100
 mobilizacao-0709        3    439   2.7           1.29  20        402            20
roberto-jefferson        1 15,972  59.1          17.50 100     11,796           100
roberto-jefferson        0  6,386  23.6          34.56 100      3,376           100
roberto-jefferson        2  4,646  17.2          17.32 100      2,535           100
         eleicoes        1  3,017  31.0          18.92 100      1,972           100
         eleicoes        0  2,788  28.6           5.64 100      2,172           100
         eleicoes        2  2,062  21.2          11.54 100      1,477           100
         eleicoes        4  1,541  15.8           5.47 100      1,301       

## Consolidação — gravação no banco e plano de hidratação

Grava a seleção de cada evento em `event_top_tweets` (*replace* por evento: o que saiu do
top-K numa re-seleção some) e deriva o plano: IDs únicos entre clusters e eventos, quantos já
estão no cache (`tweets`) e quantos faltam, com o custo pay-per-use (US$ 0,005/tweet, sem
expansions; autores à parte, US$ 0,010/user).

In [8]:
selected_at = pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")
for ev in EVENTOS:
    n = db.replace_event_top_tweets(ev, SELECOES[ev], selected_at=selected_at)
    print(f"{ev:20s} {n:4d} linhas gravadas em event_top_tweets")

ids_por_evento = {ev: set(SELECOES[ev]["tweet_id"]) for ev in EVENTOS}
todos = set().union(*ids_por_evento.values())
n_slots = sum(len(SELECOES[ev]) for ev in EVENTOS)
n_por_evento = sum(len(s) for s in ids_por_evento.values())
cached = db.cached_tweet_ids()
a_hidratar = todos - cached

print(f"\nslots (cluster×rank): {n_slots}")
print(f"IDs únicos por evento: {n_por_evento}  |  únicos no total: {len(todos)}  "
      f"(repetidos entre eventos: {n_por_evento - len(todos)})")
print(f"já no cache: {len(todos & cached)}  |  a hidratar: {len(a_hidratar)}  "
      f"≈ US$ {len(a_hidratar) * COST_PER_TWEET:.2f} em tweets")
print(f"autores: estimativa ≈ US$ {0.80 * len(a_hidratar) * COST_PER_USER:.2f} "
      f"(razão observada de 0,80 autor/tweet na hidratação de maio)")

mobilizacao-0709      320 linhas gravadas em event_top_tweets
roberto-jefferson     300 linhas gravadas em event_top_tweets
eleicoes              420 linhas gravadas em event_top_tweets
invasao-3-poderes     300 linhas gravadas em event_top_tweets

slots (cluster×rank): 1340
IDs únicos por evento: 873  |  únicos no total: 873  (repetidos entre eventos: 0)
já no cache: 76  |  a hidratar: 797  ≈ US$ 3.98 em tweets
autores: estimativa ≈ US$ 6.38 (razão observada de 0,80 autor/tweet na hidratação de maio)


## Módulo 9 — Hidratação *(a definir)*

Ainda **não implementado** — será desenhado na revisão do plano. O que já está decidido (D6):
ler do banco os IDs selecionados que não estão em `tweets`, chamar `/2/tweets` em lotes de 100
**sem expansions**, fazer *upsert* do retorno em `tweets` e registrar, por cluster, os IDs que a
API não devolveu (atrição não aleatória — tweets removidos, contas suspensas/protegidas).
O `fetch_x_data.py` atual grava JSONL com checkpoint em arquivo e descarta o array `errors`
da API; a reescrita para usar o banco como cache é a pendência aberta.

A célula abaixo só consulta o banco: é a pendência de hidratação derivada da fonte de verdade.

In [9]:
# Pendência de hidratação derivada do banco: selecionados em event_top_tweets sem linha em tweets.
pendencia = pd.read_sql("""
    SELECT e.event_slug AS evento,
           COUNT(*)                                        AS ids_selecionados,
           SUM(CASE WHEN t.tweet_id IS NULL THEN 1 ELSE 0 END) AS sem_cache
    FROM (SELECT DISTINCT event_slug, tweet_id FROM event_top_tweets) e
    LEFT JOIN tweets t ON t.tweet_id = e.tweet_id
    GROUP BY e.event_slug ORDER BY e.event_slug""", db.conn)
print(pendencia.to_string(index=False))
total = db.conn.execute("""
    SELECT COUNT(DISTINCT e.tweet_id) FROM event_top_tweets e
    LEFT JOIN tweets t ON t.tweet_id = e.tweet_id WHERE t.tweet_id IS NULL""").fetchone()[0]
print(f"\nIDs únicos a hidratar (todos os eventos): {total}  ≈ US$ {total * COST_PER_TWEET:.2f}")

           evento  ids_selecionados  sem_cache
         eleicoes               181        181
invasao-3-poderes               251        175
 mobilizacao-0709               224        224
roberto-jefferson               217        217

IDs únicos a hidratar (todos os eventos): 797  ≈ US$ 3.98


## Status — banco e artefatos

In [10]:
print("Linhas por tabela:")
for t, n in db.table_counts().items():
    print(f"  {t:24s} {n:>6,}")
print("\nArtefatos da fase 2 por evento:")
for ev in EVENTOS:
    for f in ("retweets.parquet", "top_tweets.parquet", "top_tweets_stats.json"):
        p = PROCESSED[ev] / f
        print(f"  {ev:20s} {f:24s} {p.stat().st_size / 1024:>9,.1f} KB" if p.exists() else f"  {ev:20s} {f:24s} AUSENTE")
db.close()

Linhas por tabela:
  author_classification         0
  community_membership          0
  event_top_tweets          1,340
  events                        4
  tweets                       83
  users                        72

Artefatos da fase 2 por evento:
  mobilizacao-0709     retweets.parquet           3,496.2 KB
  mobilizacao-0709     top_tweets.parquet            12.0 KB
  mobilizacao-0709     top_tweets_stats.json          1.1 KB
  roberto-jefferson    retweets.parquet          11,739.5 KB
  roberto-jefferson    top_tweets.parquet            12.3 KB
  roberto-jefferson    top_tweets_stats.json          0.9 KB
  eleicoes             retweets.parquet           3,845.3 KB
  eleicoes             top_tweets.parquet            11.3 KB
  eleicoes             top_tweets_stats.json          1.3 KB
  invasao-3-poderes    retweets.parquet          15,777.7 KB
  invasao-3-poderes    top_tweets.parquet            13.0 KB
  invasao-3-poderes    top_tweets_stats.json          0.9 KB
